# ML Enhancement: Promotion Effectiveness Analysis

Analyzes price elasticity and promotional lift to optimize marketing spend.

## Business Objective
- Identify which promotions drive incremental sales
- Understand price sensitivity by product category
- Measure promotional ROI and cannibalization effects
- Optimize discount levels and promotional calendar

## Method
- **Price Elasticity**: Log-log regression of quantity vs price
- **Incremental Lift**: Compare promoted vs. baseline sales periods
- **Cannibalization**: Measure sales decline in adjacent periods

## Data Flow
```
Configured Silver source tables
  --> configured Gold price elasticity output (product-level)
  --> configured Gold promotion lift output (promotion-level)
```

## Usage
Run this notebook after historical data load to establish baseline metrics.
Re-run periodically (weekly/monthly) to track promotional effectiveness trends.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timezone
import os
import numpy as np
from typing import Tuple
import mlflow


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name: str, default: str | None = None) -> str | None:
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="silver")
GOLD_DB = get_env("GOLD_DB", default="gold")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="promotion_effectiveness")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
PRODUCTS_TABLE = get_env("PRODUCTS_TABLE", default="dim_products")
PROMOTIONS_TABLE = get_env("PROMOTIONS_TABLE", default="fact_promotions")
PROMO_LINES_TABLE = get_env("PROMO_LINES_TABLE", default="fact_promo_lines")
PRICE_ELASTICITY_TABLE = get_env("PRICE_ELASTICITY_TABLE", default="price_elasticity")
PROMOTION_LIFT_TABLE = get_env("PROMOTION_LIFT_TABLE", default="promotion_lift")
PRICE_ELASTICITY_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRICE_ELASTICITY_TABLE}"
PROMOTION_LIFT_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PROMOTION_LIFT_TABLE}"

# Analysis parameters
MIN_OBSERVATIONS = int(get_env("MIN_OBSERVATIONS", default="30"))  # Minimum data points for elasticity calculation
BASELINE_WINDOW_DAYS = int(get_env("BASELINE_WINDOW_DAYS", default="30"))  # Days before/after promo for baseline
PROMO_EPISODE_GAP_DAYS = int(get_env("PROMO_EPISODE_GAP_DAYS", default="7"))  # Gap threshold to split reused promo codes into distinct episodes
TOP_N_PRODUCTS = int(get_env("TOP_N_PRODUCTS", default="100"))  # Top products by revenue to analyze



print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {RECEIPT_LINES_TABLE}, {PRODUCTS_TABLE}, {PROMOTIONS_TABLE}")
print(f"Promotion mapping table: {PROMO_LINES_TABLE} (optional, preferred)")
print(f"Output tables: {PRICE_ELASTICITY_TABLE_NAME}, {PROMOTION_LIFT_TABLE_NAME}")
print(f"Analysis Parameters:")
print(f"  MIN_OBSERVATIONS: {MIN_OBSERVATIONS}")
print(f"  BASELINE_WINDOW_DAYS: {BASELINE_WINDOW_DAYS}")
print(f"  PROMO_EPISODE_GAP_DAYS: {PROMO_EPISODE_GAP_DAYS}")
print(f"  TOP_N_PRODUCTS: {TOP_N_PRODUCTS}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def ensure_database(name: str) -> None:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name: str):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def save_gold(df, table_name: str) -> None:
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"  {full_name}: {df.count()} rows")

def silver_exists(table_name: str) -> bool:
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def resolve_table_column(df, table_name: str, *candidates: str) -> str:
    columns_by_lower = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        resolved = columns_by_lower.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}. Available columns: {df.columns}"
    )

ensure_database(GOLD_DB)

mlflow.set_experiment(EXPERIMENT_NAME)


---
## Part 1: Data Preparation

Join receipt lines with product dimensions and promotion data.

In [ ]:
print("="*60)
print("DATA PREPARATION")
print("="*60)

# Check required tables exist
required_tables = [RECEIPT_LINES_TABLE, PRODUCTS_TABLE, PROMOTIONS_TABLE]
missing_tables = [t for t in required_tables if not silver_exists(t)]

if missing_tables:
    print(f"ERROR: Missing required tables: {missing_tables}")
    print("Cannot proceed with analysis.")
    raise Exception(f"Missing required tables: {missing_tables}")

print("All required tables present.")

In [ ]:
# Load dimension and fact tables
df_receipt_lines = read_silver(RECEIPT_LINES_TABLE)
df_products_raw = read_silver(PRODUCTS_TABLE)
product_id_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_id", "id", "ID")
product_name_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_name", "ProductName")
product_category_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_category", "category", "Category")
product_subcategory_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_subcategory", "subcategory", "Subcategory")
product_regular_price_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "regular_price", "sale_price", "SalePrice")
product_cost_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_cost", "cost", "Cost")
df_products = df_products_raw.select(
    F.col(product_id_col).alias("product_id"),
    F.col(product_name_col).alias("product_name"),
    F.col(product_category_col).alias("product_category"),
    F.col(product_subcategory_col).alias("product_subcategory"),
    F.col(product_regular_price_col).cast("double").alias("regular_price"),
    F.col(product_cost_col).cast("double").alias("product_cost")
)
df_promotions_raw = read_silver(PROMOTIONS_TABLE)
promotion_columns_by_lower = {col.lower(): col for col in df_promotions_raw.columns}
promotion_event_ts_col = resolve_table_column(df_promotions_raw, PROMOTIONS_TABLE, "event_ts", "EventTS")
promotion_receipt_id_col = resolve_table_column(
    df_promotions_raw,
    PROMOTIONS_TABLE,
    "receipt_id",
    "ReceiptId",
    "receipt_id_ext",
    "ReceiptIdExt",
)
promotion_promo_code_col = resolve_table_column(df_promotions_raw, PROMOTIONS_TABLE, "promo_code", "PromoCode")
promotion_discount_type_col = resolve_table_column(
    df_promotions_raw,
    PROMOTIONS_TABLE,
    "discount_type",
    "DiscountType",
)
promotion_discount_cents_col = promotion_columns_by_lower.get("discount_cents") or promotion_columns_by_lower.get("discountcents")
promotion_discount_amount_col = promotion_columns_by_lower.get("discount_amount") or promotion_columns_by_lower.get("discountamount")
if promotion_discount_cents_col is None and promotion_discount_amount_col is None:
    raise RuntimeError(
        f"Unable to resolve discount amount columns in {LAKEHOUSE_NAME}.{SILVER_DB}.{PROMOTIONS_TABLE}. Available columns: {df_promotions_raw.columns}"
    )
promotion_discount_expr = (
    F.col(promotion_discount_cents_col).cast("double") / F.lit(100.0)
    if promotion_discount_cents_col is not None
    else F.col(promotion_discount_amount_col).cast("double")
)
df_promotions = df_promotions_raw.select(
    F.to_date(F.col(promotion_event_ts_col).cast("timestamp")).alias("promo_date"),
    F.col(promotion_receipt_id_col).alias("receipt_id"),
    F.col(promotion_promo_code_col).alias("promo_code"),
    F.col(promotion_discount_type_col).alias("discount_type"),
    promotion_discount_expr.alias("discount_amount"),
).cache()

print(f"Receipt lines: {df_receipt_lines.count():,} rows")
print(f"Products: {df_products.count():,} rows")
print(f"Promotions: {df_promotions.count():,} rows")

In [ ]:
# Enrich receipt lines with product attributes
df_sales = (
    df_receipt_lines
    .join(df_products, on="product_id", how="inner")
    .select(
        F.to_date(F.col("event_ts").cast("timestamp")).alias("date"),
        "receipt_id_ext",
        "product_id",
        "product_name",
        "product_category",
        "product_subcategory",
        "quantity",
        (F.col("unit_cents") / F.lit(100.0)).alias("unit_price"),
        (F.col("ext_cents") / F.lit(100.0)).alias("ext_price"),
        "promo_code",
        "regular_price",
        "product_cost"
    )
    .withColumn(
        "is_promoted",
        F.when(F.col("promo_code").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "discount_pct",
        F.when(
            (F.col("regular_price") > 0) & (F.col("unit_price") < F.col("regular_price")),
            ((F.col("regular_price") - F.col("unit_price")) / F.col("regular_price")) * 100
        ).otherwise(0.0)
    )
).cache()

print(f"\nEnriched sales data: {df_sales.count():,} rows")
df_sales.printSchema()

---
## Part 2: Price Elasticity Estimation

Calculate price elasticity using log-log regression:
```
log(quantity) = β₀ + β₁ * log(price) + ε
```
Where β₁ is the price elasticity coefficient.

In [ ]:
print("="*60)
print("PRICE ELASTICITY ANALYSIS")
print("="*60)

# Identify top products by revenue for analysis
df_top_products = (
    df_sales
    .groupBy("product_id", "product_name", "product_category")
    .agg(
        F.sum("ext_price").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
        F.countDistinct("date").alias("days_sold")
    )
    .filter(F.col("days_sold") >= MIN_OBSERVATIONS)
    .orderBy(F.desc("total_revenue"))
    .limit(TOP_N_PRODUCTS)
)

print(f"\nAnalyzing top {TOP_N_PRODUCTS} products with >= {MIN_OBSERVATIONS} days of sales data")
df_top_products.show(10, truncate=False)

In [ ]:
# Aggregate sales by product and date for elasticity calculation
df_daily_sales = (
    df_sales
    .join(
        df_top_products.select("product_id"),
        on="product_id",
        how="inner"
    )
    .groupBy("product_id", "product_name", "product_category", "date")
    .agg(
        F.sum("quantity").alias("daily_quantity"),
        F.avg("unit_price").alias("avg_price"),
        F.max("regular_price").alias("regular_price"),
        F.max("is_promoted").alias("is_promoted")
    )
    .filter(
        (F.col("daily_quantity") > 0) &
        (F.col("avg_price") > 0)
    )
    .withColumn("log_quantity", F.log(F.col("daily_quantity")))
    .withColumn("log_price", F.log(F.col("avg_price")))
)

print(f"\nDaily sales aggregation: {df_daily_sales.count():,} product-day observations")

In [ ]:
# Calculate elasticity statistics per product using aggregation
# Elasticity = covariance(log_price, log_quantity) / variance(log_price)

df_elasticity_stats = (
    df_daily_sales
    .groupBy("product_id", "product_name", "product_category")
    .agg(
        F.count("*").alias("n_observations"),
        F.avg("log_price").alias("mean_log_price"),
        F.avg("log_quantity").alias("mean_log_quantity"),
        F.stddev("log_price").alias("stddev_log_price"),
        F.stddev("log_quantity").alias("stddev_log_quantity"),
        F.avg("avg_price").alias("avg_price"),
        F.avg("regular_price").alias("regular_price"),
        F.avg("daily_quantity").alias("avg_daily_quantity")
    )
)

print(f"\nElasticity statistics calculated for {df_elasticity_stats.count():,} products")

In [ ]:
# Join back to calculate covariance and elasticity
elasticity_join_keys = ["product_id", "product_name", "product_category"]
df_elasticity_calc = (
    df_daily_sales.alias("sales")
    .join(
        df_elasticity_stats.select(
            *elasticity_join_keys,
            F.col("n_observations").alias("stats_n_observations"),
            F.col("mean_log_price").alias("stats_mean_log_price"),
            F.col("mean_log_quantity").alias("stats_mean_log_quantity"),
            F.col("stddev_log_price").alias("stats_stddev_log_price"),
            F.col("stddev_log_quantity").alias("stats_stddev_log_quantity"),
            F.col("avg_price").alias("stats_avg_price"),
            F.col("regular_price").alias("stats_regular_price"),
            F.col("avg_daily_quantity").alias("stats_avg_daily_quantity")
        ).alias("stats"),
        on=elasticity_join_keys,
        how="inner"
    )
    .withColumn(
        "price_deviation",
        F.col("log_price") - F.col("stats_mean_log_price")
    )
    .withColumn(
        "quantity_deviation",
        F.col("log_quantity") - F.col("stats_mean_log_quantity")
    )
    .withColumn(
        "cross_product",
        F.col("price_deviation") * F.col("quantity_deviation")
    )
    .withColumn(
        "price_sq_deviation",
        F.col("price_deviation") * F.col("price_deviation")
    )
)

df_price_elasticity = (
    df_elasticity_calc
    .groupBy("product_id", "product_name", "product_category")
    .agg(
        F.first("stats_n_observations").alias("n_observations"),
        F.first("stats_avg_price").alias("avg_price"),
        F.first("stats_regular_price").alias("regular_price"),
        F.first("stats_avg_daily_quantity").alias("avg_daily_quantity"),
        F.when(
            F.sum("price_sq_deviation") > 0,
            F.sum("cross_product") / F.sum("price_sq_deviation")
        ).otherwise(F.lit(None).cast("double")).alias("elasticity_coefficient"),
        F.first("stats_stddev_log_price").alias("stddev_log_price"),
        F.first("stats_stddev_log_quantity").alias("stddev_log_quantity")
    )
    .withColumn(
        "elasticity_category",
        F.when(F.abs(F.col("elasticity_coefficient")) > 1.5, "Highly Elastic")
         .when(F.abs(F.col("elasticity_coefficient")) > 1.0, "Elastic")
         .when(F.abs(F.col("elasticity_coefficient")) > 0.5, "Unit Elastic")
         .otherwise("Inelastic")
    )
    .withColumn(
        "standard_error",
        F.col("stddev_log_quantity") / F.sqrt(F.col("n_observations"))
    )
    .withColumn(
        "confidence_interval_lower",
        F.col("elasticity_coefficient") - (1.96 * F.col("standard_error"))
    )
    .withColumn(
        "confidence_interval_upper",
        F.col("elasticity_coefficient") + (1.96 * F.col("standard_error"))
    )
    .withColumn("computed_at", F.current_timestamp())
    .select(
        "product_id",
        "product_name",
        "product_category",
        "elasticity_coefficient",
        "elasticity_category",
        "confidence_interval_lower",
        "confidence_interval_upper",
        "n_observations",
        "avg_price",
        "regular_price",
        "avg_daily_quantity",
        "computed_at"
    )
)

print(f"\nPrice elasticity calculated for {df_price_elasticity.count():,} products")
print("\nSample results:")
df_price_elasticity.orderBy(F.abs(F.col("elasticity_coefficient")), ascending=False).show(10, truncate=False)

In [ ]:
# Save price elasticity to Gold layer
print(f"\nSaving {PRICE_ELASTICITY_TABLE_NAME}...")
save_gold(df_price_elasticity, PRICE_ELASTICITY_TABLE)

---
## Part 3: Promotion Lift Analysis

Measure incremental sales lift from promotions by comparing:
- Baseline sales (non-promoted periods)
- Promoted sales (during promotion)
- Post-promotion sales (cannibalization effect)

In [ ]:
print("="*60)
print("PROMOTION LIFT ANALYSIS")
print("="*60)

promo_lines_available = silver_exists(PROMO_LINES_TABLE)

if promo_lines_available:
    df_promo_lines_raw = read_silver(PROMO_LINES_TABLE)
    promo_lines_receipt_id_col = resolve_table_column(
        df_promo_lines_raw,
        PROMO_LINES_TABLE,
        "receipt_id",
        "receipt_id_ext",
        "ReceiptId",
        "ReceiptIdExt",
    )
    promo_lines_promo_code_col = resolve_table_column(
        df_promo_lines_raw,
        PROMO_LINES_TABLE,
        "promo_code",
        "PromoCode",
    )
    promo_lines_product_id_col = resolve_table_column(
        df_promo_lines_raw,
        PROMO_LINES_TABLE,
        "product_id",
        "ProductID",
    )
    df_promo_product_map = (
        df_promo_lines_raw
        .select(
            F.col(promo_lines_receipt_id_col).alias("receipt_id"),
            F.col(promo_lines_promo_code_col).alias("promo_code"),
            F.col(promo_lines_product_id_col).cast("long").alias("product_id"),
        )
        .filter(
            F.col("receipt_id").isNotNull()
            & F.col("promo_code").isNotNull()
            & F.col("product_id").isNotNull()
        )
        .distinct()
    )
    print(f"Using promo-product mapping from {LAKEHOUSE_NAME}.{SILVER_DB}.{PROMO_LINES_TABLE}")
else:
    df_promo_product_map = (
        df_sales
        .select(
            F.col("receipt_id_ext").alias("receipt_id"),
            F.col("promo_code"),
            F.col("product_id").cast("long").alias("product_id"),
        )
        .filter(
            F.col("receipt_id").isNotNull()
            & F.col("promo_code").isNotNull()
            & F.col("product_id").isNotNull()
        )
        .distinct()
    )
    print(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{PROMO_LINES_TABLE} not found; using receipt-line promo mapping from df_sales")

df_promo_daily = (
    df_promotions
    .filter(
        F.col("promo_code").isNotNull()
        & F.col("discount_type").isNotNull()
        & F.col("promo_date").isNotNull()
    )
    .groupBy("promo_code", "discount_type", "promo_date")
    .agg(F.countDistinct("receipt_id").alias("promoted_receipts"))
)

episode_window = Window.partitionBy("promo_code", "discount_type").orderBy("promo_date")
df_promo_daily = (
    df_promo_daily
    .withColumn("prev_promo_date", F.lag("promo_date").over(episode_window))
    .withColumn("gap_days", F.datediff(F.col("promo_date"), F.col("prev_promo_date")))
    .withColumn(
        "episode_break",
        F.when(
            F.col("prev_promo_date").isNull()
            | (F.col("gap_days") > PROMO_EPISODE_GAP_DAYS),
            1,
        ).otherwise(0),
    )
    .withColumn("promo_episode_id", F.sum("episode_break").over(episode_window))
)

df_promo_events = (
    df_promotions.alias("promo")
    .join(
        df_promo_daily.select("promo_code", "discount_type", "promo_date", "promo_episode_id").alias("episode"),
        (F.col("promo.promo_code") == F.col("episode.promo_code"))
        & (F.col("promo.discount_type") == F.col("episode.discount_type"))
        & (F.col("promo.promo_date") == F.col("episode.promo_date")),
        how="inner",
    )
    .select(
        F.col("promo.promo_code").alias("promo_code"),
        F.col("promo.discount_type").alias("discount_type"),
        F.col("promo.promo_date").alias("promo_date"),
        F.col("promo.receipt_id").alias("receipt_id"),
        F.col("promo.discount_amount").alias("discount_amount"),
        F.col("episode.promo_episode_id").alias("promo_episode_id"),
    )
)

df_promo_periods = (
    df_promo_events
    .groupBy("promo_code", "discount_type", "promo_episode_id")
    .agg(
        F.min("promo_date").alias("promo_start_date"),
        F.max("promo_date").alias("promo_end_date"),
        F.countDistinct("receipt_id").alias("promoted_receipts"),
        F.avg("discount_amount").alias("avg_discount"),
    )
    .withColumn("baseline_start_date", F.date_sub(F.col("promo_start_date"), BASELINE_WINDOW_DAYS))
    .withColumn("baseline_end_date", F.date_sub(F.col("promo_start_date"), 1))
    .withColumn("post_promo_start_date", F.date_add(F.col("promo_end_date"), 1))
    .withColumn("post_promo_end_date", F.date_add(F.col("promo_end_date"), BASELINE_WINDOW_DAYS))
    .cache()
)

promo_period_count = df_promo_periods.count()
print(f"\nPromotion episodes identified: {promo_period_count:,}")
df_promo_periods.show(10, truncate=False)

In [ ]:
# Map promotion episodes to products and build daily product sales facts
df_promo_events_with_products = (
    df_promo_events.alias("evt")
    .join(
        df_promo_product_map.alias("map"),
        (F.col("evt.receipt_id") == F.col("map.receipt_id"))
        & (F.col("evt.promo_code") == F.col("map.promo_code")),
        how="inner",
    )
    .select(
        F.col("evt.promo_code").alias("promo_code"),
        F.col("evt.discount_type").alias("discount_type"),
        F.col("evt.promo_episode_id").alias("promo_episode_id"),
        F.col("evt.promo_date").alias("promo_date"),
        F.col("evt.receipt_id").alias("receipt_id"),
        F.col("map.product_id").alias("product_id"),
    )
    .distinct()
)

episode_key_cols = ["promo_code", "discount_type", "promo_episode_id", "product_id"]

df_episode_products = (
    df_promo_events_with_products
    .groupBy(*episode_key_cols)
    .agg(F.countDistinct("receipt_id").alias("promoted_receipts_product"))
    .join(df_promo_periods, on=["promo_code", "discount_type", "promo_episode_id"], how="inner")
    .cache()
)

df_sales_daily_all = (
    df_sales
    .groupBy("product_id", "date")
    .agg(
        F.sum("quantity").alias("daily_quantity"),
        F.sum("ext_price").alias("daily_revenue"),
    )
    .cache()
)

df_product_promo_dates = (
    df_promo_events_with_products
    .select(F.col("product_id"), F.col("promo_date").alias("date"))
    .distinct()
    .cache()
)

episode_product_count = df_episode_products.count()
print(f"\nPromotion episode products identified: {episode_product_count:,}")

In [ ]:
# Calculate baseline sales (pre-promotion) using all sales for promoted products
def aggregate_window_sales(start_col: str, end_col: str, prefix: str, exclude_promoted_days: bool = False):
    df_window_sales = (
        df_episode_products.alias("episode")
        .join(
            df_sales_daily_all.alias("sales"),
            (F.col("sales.product_id") == F.col("episode.product_id"))
            & (F.col("sales.date") >= F.col(f"episode.{start_col}"))
            & (F.col("sales.date") <= F.col(f"episode.{end_col}")),
            how="inner",
        )
        .select(
            F.col("episode.promo_code").alias("promo_code"),
            F.col("episode.discount_type").alias("discount_type"),
            F.col("episode.promo_episode_id").alias("promo_episode_id"),
            F.col("episode.product_id").alias("product_id"),
            F.col("sales.date").alias("date"),
            F.col("sales.daily_quantity").alias("daily_quantity"),
            F.col("sales.daily_revenue").alias("daily_revenue"),
        )
    )

    if exclude_promoted_days:
        df_window_sales = df_window_sales.join(df_product_promo_dates, on=["product_id", "date"], how="left_anti")

    return (
        df_window_sales
        .groupBy(*episode_key_cols)
        .agg(
            F.sum("daily_quantity").alias(f"{prefix}_quantity"),
            F.sum("daily_revenue").alias(f"{prefix}_revenue"),
            F.countDistinct("date").alias(f"{prefix}_days"),
        )
        .withColumn(
            f"{prefix}_daily_quantity",
            F.when(
                F.col(f"{prefix}_days") > 0,
                F.col(f"{prefix}_quantity") / F.col(f"{prefix}_days"),
            ).otherwise(F.lit(0.0)),
        )
    )

df_baseline_sales = aggregate_window_sales(
    start_col="baseline_start_date",
    end_col="baseline_end_date",
    prefix="baseline",
    exclude_promoted_days=True,
)

print(f"\nBaseline sales calculated: {df_baseline_sales.count():,} promo-episode-product combinations")

In [ ]:
# Calculate promoted and post-promotion sales, plus episode-specific discount levels
df_promoted_sales = aggregate_window_sales(
    start_col="promo_start_date",
    end_col="promo_end_date",
    prefix="promo",
    exclude_promoted_days=False,
)

print(f"\nPromoted sales calculated: {df_promoted_sales.count():,} promo-episode-product combinations")

df_post_promo_sales = aggregate_window_sales(
    start_col="post_promo_start_date",
    end_col="post_promo_end_date",
    prefix="post",
    exclude_promoted_days=True,
)

print(f"\nPost-promotion sales calculated: {df_post_promo_sales.count():,} promo-episode-product combinations")

df_episode_discount_pct = (
    df_sales
    .select(
        F.col("receipt_id_ext").alias("receipt_id"),
        F.col("promo_code"),
        F.col("product_id").cast("long").alias("product_id"),
        F.col("discount_pct"),
    )
    .filter(F.col("promo_code").isNotNull())
    .join(
        df_promo_events_with_products
        .select("promo_code", "receipt_id", "product_id", "discount_type", "promo_episode_id")
        .distinct(),
        on=["promo_code", "receipt_id", "product_id"],
        how="inner",
    )
    .groupBy(*episode_key_cols)
    .agg(F.avg("discount_pct").alias("avg_discount_pct"))
)

print(f"Promotion episode discount statistics calculated: {df_episode_discount_pct.count():,} promo-episode-product combinations")

In [ ]:
# Combine and calculate lift metrics
df_promotion_lift = (
    df_episode_products
    .join(df_promoted_sales, on=episode_key_cols, how="left")
    .join(df_baseline_sales, on=episode_key_cols, how="left")
    .join(df_post_promo_sales, on=episode_key_cols, how="left")
    .join(df_episode_discount_pct, on=episode_key_cols, how="left")
    .join(
        df_products.select("product_id", "product_name", "product_category"),
        on="product_id",
        how="inner",
    )
    .select(
        "promo_code",
        "product_id",
        "product_name",
        "product_category",
        "discount_type",
        "avg_discount",
        "avg_discount_pct",
        "promo_start_date",
        "promo_end_date",
        F.coalesce(F.col("baseline_daily_quantity"), F.lit(0.0)).alias("baseline_daily_quantity"),
        F.coalesce(F.col("promo_daily_quantity"), F.lit(0.0)).alias("promo_daily_quantity"),
        F.coalesce(F.col("post_daily_quantity"), F.lit(0.0)).alias("post_daily_quantity"),
        F.coalesce(F.col("promo_quantity"), F.lit(0.0)).alias("total_promoted_quantity"),
        F.coalesce(F.col("promo_revenue"), F.lit(0.0)).alias("total_promoted_revenue"),
    )
    .withColumn(
        "incremental_lift_pct",
        F.when(
            F.col("baseline_daily_quantity") > 0,
            ((F.col("promo_daily_quantity") - F.col("baseline_daily_quantity")) /
             F.col("baseline_daily_quantity")) * 100,
        ).otherwise(F.lit(None)),
    )
    .withColumn(
        "cannibalization_pct",
        F.when(
            F.col("baseline_daily_quantity") > 0,
            ((F.col("baseline_daily_quantity") - F.col("post_daily_quantity")) /
             F.col("baseline_daily_quantity")) * 100,
        ).otherwise(F.lit(None)),
    )
    .withColumn("net_lift_pct", F.col("incremental_lift_pct") - F.col("cannibalization_pct"))
    .withColumn(
        "roi_category",
        F.when(F.col("net_lift_pct") > 50, "High ROI")
         .when(F.col("net_lift_pct") > 20, "Medium ROI")
         .when(F.col("net_lift_pct") > 0, "Low ROI")
         .otherwise("Negative ROI"),
    )
    .withColumn("computed_at", F.current_timestamp())
)

print(f"\nPromotion lift analysis complete: {df_promotion_lift.count():,} promo-episode-product combinations")
print("\nSample results:")
df_promotion_lift.orderBy(F.desc("incremental_lift_pct")).show(10, truncate=False)

In [ ]:
# Save promotion lift to Gold layer
print(f"\nSaving {PROMOTION_LIFT_TABLE_NAME}...")
save_gold(df_promotion_lift, PROMOTION_LIFT_TABLE)

---
## Summary & Insights

In [ ]:
# Log results to MLflow
with mlflow.start_run(
    run_name=f"promo_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "log_log_regression",
        "min_observations": MIN_OBSERVATIONS,
        "baseline_window_days": BASELINE_WINDOW_DAYS,
        "promo_episode_gap_days": PROMO_EPISODE_GAP_DAYS,
        "promotion_mapping_source": PROMO_LINES_TABLE if promo_lines_available else "df_sales_fallback",
        "top_n_products": TOP_N_PRODUCTS,
    })

    elasticity_count = spark.table(PRICE_ELASTICITY_TABLE_NAME).count()
    lift_count = spark.table(PROMOTION_LIFT_TABLE_NAME).count()
    mlflow.log_metrics({
        "elasticity_products": elasticity_count,
        "promotion_lift_records": lift_count,
    })

    print(f"MLflow run: {run.info.run_id}")
    print(f"Elasticity: {elasticity_count} products, Lift: {lift_count} records")


In [ ]:
print("\n" + "="*60)
print("PROMOTION EFFECTIVENESS ANALYSIS COMPLETE")
print("="*60)

# Elasticity summary
print("\n--- Price Elasticity Summary ---")
df_price_elasticity.groupBy("elasticity_category").count().orderBy(F.desc("count")).show()

print("\nMost price-sensitive products (lowest elasticity):")
df_price_elasticity.orderBy(F.asc("elasticity_coefficient")).select(
    "product_name", "product_category", "elasticity_coefficient", "elasticity_category"
).show(5, truncate=False)

print("\nLeast price-sensitive products (highest elasticity):")
df_price_elasticity.orderBy(F.desc("elasticity_coefficient")).select(
    "product_name", "product_category", "elasticity_coefficient", "elasticity_category"
).show(5, truncate=False)

# Promotion lift summary
print("\n--- Promotion Lift Summary ---")
df_promotion_lift.groupBy("roi_category").count().orderBy(F.desc("count")).show()

print("\nTop performing promotions (highest net lift):")
df_promotion_lift.orderBy(F.desc("net_lift_pct")).select(
    "promo_code", "product_name", "discount_type", 
    "incremental_lift_pct", "cannibalization_pct", "net_lift_pct", "roi_category"
).show(5, truncate=False)

print("\nPromotions with high cannibalization:")
df_promotion_lift.filter(F.col("cannibalization_pct") > 20).orderBy(F.desc("cannibalization_pct")).select(
    "promo_code", "product_name", "discount_type",
    "incremental_lift_pct", "cannibalization_pct", "net_lift_pct"
).show(5, truncate=False)

print("\nGold tables created:")
print(f"  - {PRICE_ELASTICITY_TABLE_NAME}")
print(f"  - {PROMOTION_LIFT_TABLE_NAME}")
print("\nUse these configured output tables for Power BI dashboards and further analysis.")